# Figure1c external roc


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import load

MODEL_DIR = 'results/classification_sle_external/results_elasticnet_C1_l1_0.5'

In [ ]:
def plot_roc_validation_combined(
    nyu_results_dir,
    itn_results_dir,
    save_plot=False,
    output_path=None,
    figsize=(7, 7),
    ci_level=0.95,
    decimal_places=2,
):
    nyu_color = "#90719f"
    itn_color = "#466c4b"

    print("Loading NYU validation data...")
    nyu_roc_data = load(f'{nyu_results_dir}/test_roc_data.joblib')
    nyu_results_df = pd.read_csv(f'{nyu_results_dir}/test_results.csv')
    print(f"  - {len(nyu_roc_data)} models loaded")
    print(f"  - Mean AUC: {nyu_results_df['auroc'].mean():.3f} ± {nyu_results_df['auroc'].std():.3f}")

    print("\nLoading ITN validation data...")
    itn_roc_data = load(f'{itn_results_dir}/itn_test_roc_data.joblib')
    itn_results_df = pd.read_csv(f'{itn_results_dir}/itn_test_results.csv')
    print(f"  - {len(itn_roc_data)} models loaded")
    print(f"  - Mean AUC: {itn_results_df['auroc'].mean():.3f} ± {itn_results_df['auroc'].std():.3f}")

    mean_fpr = np.linspace(0, 1, 1000)

    tprs_nyu = []
    for data in nyu_roc_data:
        interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
        interp_tpr[0] = 0.0
        tprs_nyu.append(interp_tpr)
    tprs_nyu = np.array(tprs_nyu)
    mean_tpr_nyu = np.mean(tprs_nyu, axis=0)
    mean_tpr_nyu[-1] = 1.0

    tprs_itn = []
    for data in itn_roc_data:
        interp_tpr = np.interp(mean_fpr, data['fpr'], data['tpr'])
        interp_tpr[0] = 0.0
        tprs_itn.append(interp_tpr)
    tprs_itn = np.array(tprs_itn)
    mean_tpr_itn = np.mean(tprs_itn, axis=0)
    mean_tpr_itn[-1] = 1.0

    alpha_low = (1 - ci_level) / 2 * 100
    alpha_high = (1 + ci_level) / 2 * 100

    tprs_lower_nyu = np.percentile(tprs_nyu, alpha_low, axis=0)
    tprs_upper_nyu = np.percentile(tprs_nyu, alpha_high, axis=0)
    tprs_lower_itn = np.percentile(tprs_itn, alpha_low, axis=0)
    tprs_upper_itn = np.percentile(tprs_itn, alpha_high, axis=0)

    aucs_nyu = nyu_results_df['auroc'].values
    mean_auc_nyu = np.mean(aucs_nyu)
    ci_low_auc_nyu = np.percentile(aucs_nyu, alpha_low)
    ci_high_auc_nyu = np.percentile(aucs_nyu, alpha_high)

    aucs_itn = itn_results_df['auroc'].values
    mean_auc_itn = np.mean(aucs_itn)
    ci_low_auc_itn = np.percentile(aucs_itn, alpha_low)
    ci_high_auc_itn = np.percentile(aucs_itn, alpha_high)

    fig, ax = plt.subplots(figsize=figsize)
    fmt = f'.{decimal_places}f'

    nyu_label = f'NYU cohort: AUC = {mean_auc_nyu:{fmt}} ({ci_low_auc_nyu:{fmt}}–{ci_high_auc_nyu:{fmt}})'
    ax.fill_between(mean_fpr, tprs_lower_nyu, tprs_upper_nyu,
                    color=nyu_color, alpha=0.15, linewidth=0, zorder=2)
    ax.plot(mean_fpr, mean_tpr_nyu, color=nyu_color, linewidth=2.5,
            label=nyu_label, zorder=4)

    itn_label = f'ITN cohort: AUC = {mean_auc_itn:{fmt}} ({ci_low_auc_itn:{fmt}}–{ci_high_auc_itn:{fmt}})'
    ax.fill_between(mean_fpr, tprs_lower_itn, tprs_upper_itn,
                    color=itn_color, alpha=0.15, linewidth=0, zorder=2)
    ax.plot(mean_fpr, mean_tpr_itn, color=itn_color, linewidth=2.5,
            label=itn_label, zorder=4)

    ax.plot([0, 1], [0, 1], color='gray', linewidth=1.5, linestyle='--',
            alpha=0.7, label='Random classifier', zorder=1)

    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_xlabel('1 − Specificity', fontsize=15, fontweight='medium')
    ax.set_ylabel('Sensitivity', fontsize=15, fontweight='medium')
    ax.tick_params(axis='both', which='major', labelsize=12, length=5)
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, zorder=0)
    ax.set_aspect('equal', adjustable='box')

    legend = ax.legend(loc='lower right', fontsize=13, framealpha=0.95,
                       edgecolor='gray', fancybox=False)
    legend.get_frame().set_linewidth(0.8)

    plt.tight_layout()

    if save_plot and output_path:
        plt.savefig(f'{output_path}.pdf', bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print(f"\nFigures saved to: {output_path}.pdf")

    print("\n" + "="*60)
    print("EXTERNAL VALIDATION ROC STATISTICS")
    print("="*60)
    print(f"\nNYU Validation Cohort:")
    print(f"  Mean AUC: {mean_auc_nyu:.4f}")
    print(f"  95% CI:   [{ci_low_auc_nyu:.4f}, {ci_high_auc_nyu:.4f}]")
    print(f"  N models: {len(aucs_nyu)}")
    print(f"\nITN Validation Cohort:")
    print(f"  Mean AUC: {mean_auc_itn:.4f}")
    print(f"  95% CI:   [{ci_low_auc_itn:.4f}, {ci_high_auc_itn:.4f}]")
    print(f"  N models: {len(aucs_itn)}")
    print("\n" + "="*60)

    plt.show()

    stats_dict = {
        'nyu': {'mean_auc': mean_auc_nyu, 'ci_low_auc': ci_low_auc_nyu,
                'ci_high_auc': ci_high_auc_nyu, 'n_models': len(aucs_nyu)},
        'itn': {'mean_auc': mean_auc_itn, 'ci_low_auc': ci_low_auc_itn,
                'ci_high_auc': ci_high_auc_itn, 'n_models': len(aucs_itn)}
    }
    return fig, ax, stats_dict

In [ ]:
fig, ax, stats = plot_roc_validation_combined(
    nyu_results_dir=MODEL_DIR,
    itn_results_dir=MODEL_DIR,
    output_path=f"{MODEL_DIR}/fig1c_updated_nyu_itn_test_roc",
    save_plot=True,
    figsize=(7, 7),
    ci_level=0.95,
    decimal_places=2,
)